# 🛡️ Databricks AI Gateway - Demo AT&T Mexico

**AI Gateway** es la capa de gobernanza centralizada para todas las llamadas a modelos de IA (LLMs, embeddings, etc.) en Databricks.

### ¿Qué problemas resuelve?
| Problema | Solución AI Gateway |
|----------|--------------------|
| Sin visibilidad de uso | **Inference Tables** - logs de cada request/response |
| Costos descontrolados | **Rate Limiting** - límites por usuario/endpoint |
| Datos sensibles expuestos | **Guardrails** - filtros de seguridad input/output |
| Sin auditoría | **Usage Tracking** - métricas en system tables |
| Vendor lock-in | **Multi-provider** - OpenAI, Anthropic, Google, etc. |

### Arquitectura
```
Usuario/App → AI Gateway (rate limit + guardrails + logging) → Modelo (GPT-4o, Claude, Llama, etc.)
                    ↓
         Inference Table (Unity Catalog)
         System Tables (usage, billing)
```

In [0]:
%pip install --upgrade databricks-sdk openai mlflow
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ExternalModel,
    OpenAiConfig,
    AnthropicConfig,
    DatabricksModelServingConfig,
    ExternalModelProvider,
    AiGatewayConfig,
    AiGatewayInferenceTableConfig,
    AiGatewayUsageTrackingConfig,
    AiGatewayRateLimit,
    AiGatewayRateLimitRenewalPeriod,
    AiGatewayRateLimitKey,
    AiGatewayGuardrails,
    AiGatewayGuardrailParameters,
    AiGatewayGuardrailPiiBehavior,
    AiGatewayGuardrailPiiBehaviorBehavior,
)
import json

w = WorkspaceClient()

# Parámetros de la demo
ENDPOINT_NAME = "att-mexico-ai-gateway-demo"
CATALOG = "mozuca"
SCHEMA = "ai_gateway"

print(f"✅ Conectado al workspace: {w.config.host}")
print(f"📋 Endpoint a crear: {ENDPOINT_NAME}")

## 💰 Costos de AI Gateway

AI Gateway **no tiene un costo adicional separado** — el cobro está incluido en el consumo de **Model Serving (DBUs)**.

| Componente | Costo | Detalle |
|-----------|-------|----------|
| **Foundation Models** (Llama, Claude, etc.) | Pay-per-token en DBUs | Se cobra por tokens de input/output consumidos |
| **External Models** (OpenAI, Anthropic directo) | Pass-through | Databricks no cobra extra; pagas al proveedor + DBUs mínimos de routing |
| **Guardrails** | Incluido | Sin costo adicional por evaluación de safety/PII |
| **Rate Limiting** | Incluido | Sin costo |
| **Inference Tables** | Storage en Delta | Costo estándar de almacenamiento en Unity Catalog |
| **Usage Tracking** | Incluido | System tables sin costo adicional |

### Cómo consultar costos:
```sql
-- Costo por endpoint (últimos 30 días)
SELECT
  usage_metadata.ai_gateway_endpoint_name AS endpoint,
  SUM(usage_quantity) AS dbus
FROM system.billing.usage
WHERE billing_origin_product = 'MODEL_SERVING'
  AND usage_metadata.ai_gateway_endpoint_name IS NOT NULL
  AND usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY endpoint ORDER BY dbus DESC;

-- Costo por usuario (chargeback entre áreas)
SELECT
  identity_metadata.run_by AS usuario,
  SUM(usage_quantity) AS dbus
FROM system.billing.usage
WHERE billing_origin_product = 'MODEL_SERVING'
  AND usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY usuario ORDER BY dbus DESC;
```

---

## 🔐 Permisos necesarios para crear y usar AI Gateway

### Para CREAR un endpoint con AI Gateway:
| Permiso | Quién lo necesita | Para qué |
|---------|-------------------|----------|
| **Workspace admin** o **CAN MANAGE** en serving endpoints | El creador del endpoint | Crear/configurar el endpoint |
| **Unity Catalog**: `USE CATALOG` + `USE SCHEMA` + `CREATE TABLE` | El creador | Para la Inference Table (se crea auto en UC) |
| **Secret scope access** (si usa external models) | El creador | Para almacenar API keys de OpenAI/Anthropic/etc. |

### Para USAR (consultar) un endpoint:
| Permiso | Quién lo necesita | Para qué |
|---------|-------------------|----------|
| **CAN QUERY** en el endpoint | Usuarios finales / apps | Enviar requests al modelo |
| **CAN VIEW** en el endpoint | Usuarios que solo monitorean | Ver config sin poder consultar |

### Para configurar Guardrails:
| Permiso | Quién lo necesita | Para qué |
|---------|-------------------|----------|
| **CAN MANAGE** en el endpoint de inferencia | Admin del endpoint | Configurar guardrails |
| **CAN MANAGE** en el endpoint evaluador | Admin (si usa evaluador custom) | El evaluador que ejecuta los guardrails |

### ACL de Serving Endpoints (resumen):
| Acción | CAN VIEW | CAN QUERY | CAN MANAGE |
|--------|----------|-----------|------------|
| Ver endpoint | ✓ | ✓ | ✓ |
| Consultar (enviar requests) | | ✓ | ✓ |
| Modificar configuración | | | ✓ |
| Eliminar endpoint | | | ✓ |
| Cambiar permisos | | | ✓ |

### Requisitos de cuenta:
- **Unity Catalog** habilitado en el workspace
- **AI Gateway Preview** habilitado por account admin (desde Account Console → Previews)
- Workspace en una **región soportada** para AI Gateway

In [0]:
# Crear un endpoint con modelo externo + AI Gateway habilitado
# Incluye: Rate Limiting, Guardrails, Inference Table, Usage Tracking
import time

# Verificar si el endpoint ya existe (de una ejecución previa)
try:
    existing = w.serving_endpoints.get(name=ENDPOINT_NAME)
    print(f"✅ Endpoint '{ENDPOINT_NAME}' ya existe y está {existing.state.ready}")
    print("   Saltando creación — usando endpoint existente.")
    endpoint = existing
    _skip_creation = True
except Exception:
    _skip_creation = False

if not _skip_creation:

  # Crear el endpoint con TODAS las features de AI Gateway
  endpoint = w.serving_endpoints.create(
      name=ENDPOINT_NAME,
      config=EndpointCoreConfigInput(
          name=ENDPOINT_NAME,
          served_entities=[
              ServedEntityInput(
                  name="llama-3-att",
                  external_model=ExternalModel(
                      name="databricks-meta-llama-3-3-70b-instruct",
                      provider=ExternalModelProvider.DATABRICKS_MODEL_SERVING,
                      task="llm/v1/chat",
                      databricks_model_serving_config=DatabricksModelServingConfig(
                          databricks_workspace_url=w.config.host,
                          databricks_api_token_plaintext=w.config.authenticate()["Authorization"].replace("Bearer ", ""),
                      ),
                  ),
              )
          ]
      ),
      ai_gateway=AiGatewayConfig(
          # 📊 Inference Table - guarda TODOS los requests y responses
          inference_table_config=AiGatewayInferenceTableConfig(
              catalog_name=CATALOG,
              schema_name=SCHEMA,
              table_name_prefix="att_aigw_demo_run",
              enabled=True,
          ),
          # 📈 Usage Tracking - métricas en system tables
          usage_tracking_config=AiGatewayUsageTrackingConfig(enabled=True),
          # 🚦 Rate Limits - controlar consumo
          rate_limits=[
              AiGatewayRateLimit(
                  calls=10,
                  renewal_period=AiGatewayRateLimitRenewalPeriod.MINUTE,
                  key=AiGatewayRateLimitKey.USER,
              )
          ],
          # 🛡️ Guardrails - seguridad de contenido
          guardrails=AiGatewayGuardrails(
              input=AiGatewayGuardrailParameters(
                  safety=True,
                  pii=AiGatewayGuardrailPiiBehavior(
                      behavior=AiGatewayGuardrailPiiBehaviorBehavior.BLOCK
                  ),
              ),
              output=AiGatewayGuardrailParameters(
                  safety=True,
                  pii=AiGatewayGuardrailPiiBehavior(
                      behavior=AiGatewayGuardrailPiiBehaviorBehavior.BLOCK
                  ),
              ),
          ),
      ),
  )

print(f"""\n🎉 Endpoint creado exitosamente!
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📛 Nombre:          {ENDPOINT_NAME}
🤖 Modelo:          Llama 3.3 70B (Databricks Foundation Model)
📊 Inference Table: {CATALOG}.{SCHEMA}.{ENDPOINT_NAME.replace('-','_')}_payload
📈 Usage Tracking:  Habilitado
🚦 Rate Limit:      10 requests/min/usuario
🛡️ Guardrails:      Safety + PII (input & output)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━""")

In [0]:
import time

print("⏳ Esperando a que el endpoint esté READY...")
for i in range(60):
    status = w.serving_endpoints.get(name=ENDPOINT_NAME)
    state = status.state.ready
    print(f"   Estado: {state} ({i*5}s)", end="\r")
    if str(state) == "READY" or "READY" in str(state):
        print(f"\n✅ ¡Endpoint LISTO! (tardó ~{i*5} segundos)")
        break
    time.sleep(5)
else:
    print("\n⚠️ Timeout - verificar en la UI")

In [0]:
from openai import OpenAI

# Conectar via OpenAI SDK compatible (Databricks Foundation Models)
# AI Gateway ya está habilitado por defecto en TODOS los Foundation Models
token = w.config.authenticate()["Authorization"].replace("Bearer ", "")
client = OpenAI(
    api_key=token,
    base_url=f"{w.config.host}/serving-endpoints"
)

# Modelo a usar (Foundation Model con AI Gateway integrado)
MODEL_NAME = "databricks-meta-llama-3-3-70b-instruct"

# Caso 1: Pregunta normal de negocio
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "Eres un asistente experto en telecomunicaciones para AT&T Mexico."},
        {"role": "user", "content": "¿Cuáles son las mejores prácticas para reducir el churn en clientes prepago?"}
    ],
    max_tokens=500
)

print("📨 Respuesta del modelo (via AI Gateway):")
print("="*50)
print(response.choices[0].message.content)
print(f"\n📊 Tokens usados: {response.usage.total_tokens} (input: {response.usage.prompt_tokens}, output: {response.usage.completion_tokens})")
print(f"\n💡 Esta llamada fue automáticamente registrada en system.ai_gateway.usage")

In [0]:
# Caso 2: Intentar enviar PII - el guardrail debe BLOQUEAR
try:
    response_pii = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": "El cliente Juan Pérez con CURP PEGJ850101HDFRRL09 y tarjeta 4111-1111-1111-1111 llamó para cancelar. Su número es 55-1234-5678."}
        ],
        max_tokens=200
    )
    print("⚠️ Respuesta recibida (guardrail no bloqueó):")
    print(response_pii.choices[0].message.content)
except Exception as e:
    print("🛡️ ¡GUARDRAIL ACTIVADO!")
    print(f"   El AI Gateway bloqueó la solicitud por contener PII")
    print(f"   Error: {str(e)[:200]}")
    print("\n   ✅ Esto protege datos sensibles de clientes (CURP, tarjetas, teléfonos)")

In [0]:
# Caso 3: Demostrar rate limiting (enviar muchos requests rápido)
import concurrent.futures

def make_request(i):
    try:
        r = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": f"Hola, test #{i}"}],
            max_tokens=10
        )
        return f"✅ Request #{i}: OK"
    except Exception as e:
        if "429" in str(e):
            return f"🚦 Request #{i}: RATE LIMITED (429)"
        return f"❌ Request #{i}: {str(e)[:80]}"

print("🚦 Probando Rate Limiting (enviando 15 requests rápidos)...")
print("   Límite configurado: 10 requests/minuto/usuario\n")

with concurrent.futures.ThreadPoolExecutor(max_workers=15) as executor:
    results = list(executor.map(make_request, range(1, 16)))

for r in results:
    print(f"   {r}")

blocked = sum(1 for r in results if "RATE LIMITED" in r)
print(f"\n📊 Resumen: {15-blocked} exitosos, {blocked} bloqueados por rate limit")

## 🔄 Fallback: Multi-Proveedor con Claude como respaldo

Una de las features más poderosas de AI Gateway es **fallback automático**. Si GPT-4o falla (429 rate limit, 5xx error), el request se reenvía automáticamente a Claude como respaldo.

```
Request → GPT-4o (primario, 100% tráfico)
              ↓ (si falla 429/5xx)
          Claude (fallback, 0% tráfico directo)
```

**Beneficio AT&T:** Alta disponibilidad sin cambiar código de aplicación.

In [0]:
# Demostrar fallback multi-proveedor usando Foundation Models de Databricks
# Primario: Llama 3.3 70B | Fallback: Claude Opus (Databricks-hosted)
# ✅ No requiere API keys externas - usa Foundation Model APIs

from databricks.sdk.service.serving import TrafficConfig, Route
import time

FALLBACK_ENDPOINT = "att-mexico-fallback-demo"

# Verificar si ya existe
try:
    _fb_existing = w.serving_endpoints.get(name=FALLBACK_ENDPOINT)
    print(f"✅ Endpoint '{FALLBACK_ENDPOINT}' ya existe ({_fb_existing.state.ready})")
    print("   Saltando creación — usando endpoint existente.")
    _skip_fb = True
except Exception:
    _skip_fb = False

if not _skip_fb:
  try:
      w.serving_endpoints.delete(name=FALLBACK_ENDPOINT)
      time.sleep(5)
      print(f"🗑️ Endpoint previo eliminado")
  except Exception:
      pass

  # Obtener PAT para autenticación server-to-server
  demo_pat = w.tokens.create(comment="att-fallback-demo", lifetime_seconds=7200).token_value

  fallback_ep = w.serving_endpoints.create(
    name=FALLBACK_ENDPOINT,
    config=EndpointCoreConfigInput(
        name=FALLBACK_ENDPOINT,
        served_entities=[
            # Modelo PRIMARIO - recibe 100% del tráfico (Llama 3.3)
            ServedEntityInput(
                name="llama-primary",
                external_model=ExternalModel(
                    name="databricks-meta-llama-3-3-70b-instruct",
                    provider=ExternalModelProvider.DATABRICKS_MODEL_SERVING,
                    task="llm/v1/chat",
                    databricks_model_serving_config=DatabricksModelServingConfig(
                        databricks_workspace_url=w.config.host,
                        databricks_api_token_plaintext=demo_pat,
                    ),
                ),
            ),
            # Modelo FALLBACK - solo si Llama falla (Claude Opus)
            ServedEntityInput(
                name="claude-fallback",
                external_model=ExternalModel(
                    name="databricks-claude-opus-4-6",
                    provider=ExternalModelProvider.DATABRICKS_MODEL_SERVING,
                    task="llm/v1/chat",
                    databricks_model_serving_config=DatabricksModelServingConfig(
                        databricks_workspace_url=w.config.host,
                        databricks_api_token_plaintext=demo_pat,
                    ),
                ),
            ),
        ],
        traffic_config=TrafficConfig(
            routes=[
                Route(served_model_name="llama-primary", traffic_percentage=100),
                Route(served_model_name="claude-fallback", traffic_percentage=0),
            ]
        ),
    ),
    ai_gateway=AiGatewayConfig(
        inference_table_config=AiGatewayInferenceTableConfig(
            catalog_name=CATALOG,
            schema_name=SCHEMA,
            table_name_prefix="att_fb_demo_run",
            enabled=True,
        ),
        usage_tracking_config=AiGatewayUsageTrackingConfig(enabled=True),
        rate_limits=[
            AiGatewayRateLimit(
                calls=10,
                renewal_period=AiGatewayRateLimitRenewalPeriod.MINUTE,
                key=AiGatewayRateLimitKey.USER,
            )
        ],
        guardrails=AiGatewayGuardrails(
            input=AiGatewayGuardrailParameters(safety=True),
            output=AiGatewayGuardrailParameters(safety=True),
        ),
    ),
)

  print(f"""
🎉 Endpoint con FALLBACK creado!
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📛 Nombre:       {FALLBACK_ENDPOINT}
🤖 Primario:     Llama 3.3 70B (Meta) → 100% del tráfico
🔄 Fallback:     Claude Opus (Anthropic) → 0% directo, solo si Llama falla
🛡️ Guardrails:   Safety habilitado
📊 Logging:      Inference table + usage tracking
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔄 Comportamiento de Fallback:
   1. Request llega → se envía a Llama 3.3
   2. Si Llama responde 200 → éxito, se loguea
   3. Si Llama responde 429/5xx → se reenvía a Claude Opus
   4. Si Claude responde 200 → éxito, se loguea
   5. Si ambos fallan → error al cliente

💡 El cliente NO necesita cambiar su código - el fallback es transparente.
""")

In [0]:
# Demostrar el concepto de fallback multi-modelo
# Llamamos a ambos Foundation Models directamente para mostrar que ambos funcionan
# En producción con API keys externas (OpenAI/Anthropic), el fallback es automático

import time

# Verificar que el fallback endpoint está READY
print("⏳ Verificando estado del endpoint fallback...")
status = w.serving_endpoints.get(name=FALLBACK_ENDPOINT)
print(f"✅ Estado: {status.state.ready}")
print(f"   Entidades configuradas:")
for entity in status.config.served_entities:
    print(f"   • {entity.name} → {entity.external_model.name}")

# Demostrar que ambos modelos responden (simulando el comportamiento de fallback)
print("\n" + "="*50)
print("🔄 DEMO: Ambos modelos del fallback responden correctamente")
print("="*50)

# Modelo primario: Llama 3.3
print("\n🤖 [PRIMARIO] Llama 3.3 70B:")
resp_primary = client.chat.completions.create(
    model="databricks-meta-llama-3-3-70b-instruct",
    messages=[
        {"role": "system", "content": "Eres un experto en telecomunicaciones. Responde en 1-2 oraciones."},
        {"role": "user", "content": "¿Qué ventajas tiene el 5G para IoT empresarial?"}
    ],
    max_tokens=100
)
print(f"   {resp_primary.choices[0].message.content}")
print(f"   📊 Tokens: {resp_primary.usage.total_tokens}")

# Modelo fallback: Claude Opus
print("\n🔄 [FALLBACK] Claude Opus:")
resp_fallback = client.chat.completions.create(
    model="databricks-claude-opus-4-6",
    messages=[
        {"role": "system", "content": "Eres un experto en telecomunicaciones. Responde en 1-2 oraciones."},
        {"role": "user", "content": "¿Qué ventajas tiene el 5G para IoT empresarial?"}
    ],
    max_tokens=100
)
print(f"   {resp_fallback.choices[0].message.content}")
print(f"   📊 Tokens: {resp_fallback.usage.total_tokens}")

print(f"""\n
💡 En el endpoint '{FALLBACK_ENDPOINT}':
   • Si Llama 3.3 falla (429/5xx) → AI Gateway reenvía automáticamente a Claude
   • El cliente recibe la respuesta SIN saber que hubo un failover
   • Todo queda registrado en system.ai_gateway.usage (campo destination_name)
""")

In [0]:
%sql
-- 🔄 Verificar qué modelo respondió cada request (routing/fallback)
-- El campo destination_name muestra si fue el primario o el fallback

SELECT 
  date_format(event_time, 'HH:mm:ss') as hora,
  endpoint_name,
  destination_name as modelo_que_respondio,
  status_code,
  input_tokens,
  output_tokens,
  total_tokens
FROM system.ai_gateway.usage
WHERE endpoint_name = 'att-mexico-fallback-demo'
  AND event_time > current_date() - INTERVAL 1 DAY
ORDER BY event_time DESC
LIMIT 10

In [0]:
# Generar algunos requests adicionales para tener datos en inference table
import time

preguntas = [
    "¿Qué plan de datos recomiendas para una familia de 4?",
    "¿Cómo puedo mejorar la cobertura 5G en zonas rurales?",
    "Resume las tendencias del mercado de telecomunicaciones en México 2024",
    "¿Cuál es la diferencia entre fibra óptica y cable coaxial?",
]

print("📨 Enviando requests adicionales para generar logs...\n")
for i, pregunta in enumerate(preguntas, 1):
    time.sleep(7)  # Respetar rate limit
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "Eres un experto en telecomunicaciones. Responde en español de forma concisa."},
            {"role": "user", "content": pregunta}
        ],
        max_tokens=200
    )
    print(f"   ✅ [{i}/{len(preguntas)}] {pregunta[:50]}...")
    print(f"      → {response.choices[0].message.content[:100]}...\n")

print("\n⏳ Esperando 60s para que los logs se materialicen en la inference table...")
time.sleep(60)

In [0]:
%sql
-- 📊 INFERENCE TABLE: Contiene TODOS los requests y responses
-- Se crea automáticamente en Unity Catalog al habilitar AI Gateway
-- ⚠️ Los datos tardan ~10-30 min en materializarse después de las llamadas
-- Una vez poblada, contendrá columnas como:
--   databricks_request_id, request, response, status_code, 
--   timestamp_ms, execution_time_ms, request_metadata

-- Verificar que la tabla existe y su schema actual:
DESCRIBE TABLE mozuca.ai_gateway.att_mexico_ai_gateway_demo_payload

In [0]:
%sql
-- 📈 SYSTEM TABLE: Métricas de uso agregadas (disponible para admins)
-- Muestra uso por endpoint, usuario, tokens, costos estimados

SELECT 
  date_format(event_time, 'yyyy-MM-dd HH:mm') as hora,
  endpoint_name,
  requester as usuario,
  status_code,
  input_tokens,
  output_tokens,
  total_tokens
FROM system.ai_gateway.usage
WHERE endpoint_name = 'databricks-meta-llama-3-3-70b-instruct'
  AND event_time > current_date() - INTERVAL 1 DAY
  AND requester LIKE '%moises.santos%'
ORDER BY event_time DESC
LIMIT 20

In [0]:
# Resumen visual del uso del endpoint
df_usage = spark.sql(f"""
    SELECT 
        date_trunc('minute', event_time) as minuto,
        count(*) as total_requests,
        sum(CASE WHEN status_code = 200 THEN 1 ELSE 0 END) as exitosos,
        sum(CASE WHEN status_code = 429 THEN 1 ELSE 0 END) as rate_limited,
        sum(CASE WHEN status_code = 400 THEN 1 ELSE 0 END) as bloqueados_guardrail,
        sum(total_tokens) as total_tokens,
        avg(total_tokens) as avg_tokens_por_request
    FROM system.ai_gateway.usage
    WHERE endpoint_name = 'databricks-meta-llama-3-3-70b-instruct'
      AND requester LIKE '%moises.santos%'
      AND event_time > current_date() - INTERVAL 1 DAY
    GROUP BY 1
    ORDER BY 1
""")

print("📊 Resumen de uso del AI Gateway")
print("="*50)
display(df_usage)

## 📋 Resumen: Valor de AI Gateway para AT&T Mexico

| Feature | Qué hace | Valor para AT&T |
|---------|----------|------------------|
| **Inference Table** | Guarda cada request/response con timestamps, tokens, latencia | Auditoría completa de uso de IA, debugging, compliance |
| **Rate Limiting** | Limita requests por usuario/minuto | Control de costos, prevenir abuso, capacity planning |
| **Guardrails (PII)** | Bloquea datos personales (CURP, tarjetas, teléfonos) | Cumplimiento LFPDPPP, protección de datos de clientes |
| **Guardrails (Safety)** | Filtra contenido inseguro/inapropiado | Brand safety, uso responsable de IA |
| **Usage Tracking** | Métricas en system tables de Databricks | Dashboards de consumo, chargeback entre áreas |
| **Multi-provider** | Misma interfaz para OpenAI, Anthropic, Google, etc. | Sin vendor lock-in, fácil cambiar proveedor |

### Información que guarda AI Gateway:
1. **Request completo** - quién preguntó, qué preguntó, cuándo
2. **Response completo** - qué respondió el modelo, tokens usados
3. **Metadata** - latencia, status code, modelo usado
4. **Seguridad** - requests bloqueados por guardrails, rate limits
5. **Costos** - tokens consumidos para estimar gastos

### Próximos pasos sugeridos:
- Conectar un dashboard para monitoreo en tiempo real
- Configurar alertas cuando se exceda uso
- Integrar con flujos de MLOps existentes
- Evaluar calidad de respuestas con MLflow

In [0]:
# ⚠️ Descomentar para eliminar el endpoint después de la demo
# w.serving_endpoints.delete(name=ENDPOINT_NAME)
# print(f"🗑️ Endpoint '{ENDPOINT_NAME}' eliminado")